# Train trên Kaggle hoặc máy cục bộ
Bật GPU trong Kaggle Settings, upload source project và dữ liệu vào working/input. Cài `pip install -r requirements.txt`, dùng PyTorch/CUDA có sẵn nếu tương thích. Notebook mặc định **smoke test**, không dùng để công bố chất lượng.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/pipeline.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Đặt working directory trong project đã giải nén/clone.")
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [ ]:
import subprocess, yaml, torch
print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
FULL_TRAINING = False
TASK = "shadow_deblur"  # hoặc "warp"
CLEAN_ROOT = Path("/kaggle/input/your-clean-documents") if FULL_TRAINING else Path("experiments/notebook_smoke/clean")
WORK = Path("/kaggle/working/doc-restore-runs") if Path("/kaggle").exists() else Path("experiments/notebook_smoke")
WORK.mkdir(parents=True, exist_ok=True)
if not FULL_TRAINING and not CLEAN_ROOT.exists():
    subprocess.run([sys.executable, "-m", "scripts.make_demo_data", "--output", str(CLEAN_ROOT)], check=True)
DATA_ROOT = WORK/TASK
module = "scripts.make_dataset" if TASK == "shadow_deblur" else "scripts.make_warp_dataset"
if not (DATA_ROOT/"manifest.json").exists():
    subprocess.run([sys.executable,"-m",module,"--clean",str(CLEAN_ROOT),"--output",str(DATA_ROOT),
                    "--variants", "8" if FULL_TRAINING else "1", "--size", "512" if FULL_TRAINING else "128"],check=True)


In [ ]:
config_path = Path("configs/shadow_deblur.yaml" if TASK == "shadow_deblur" else "configs/dewarp.yaml")
config = yaml.safe_load(config_path.read_text())
config["data_root"] = str(DATA_ROOT)
config["output_dir"] = str(WORK/"checkpoints"/TASK)
if not FULL_TRAINING:
    config.update(epochs=1, batch_size=2, num_workers=0, image_size=64)
    if TASK == "shadow_deblur": config["model"]["base"] = 4
local_config = WORK/f"{TASK}.yaml"
local_config.write_text(yaml.safe_dump(config))
trainer = "scripts.train_shadow_deblur" if TASK == "shadow_deblur" else "scripts.train_dewarp"
command = [sys.executable,"-m",trainer,"--config",str(local_config)]
if not FULL_TRAINING: command += ["--max-batches", "2"]
# Chạy lại cell cần --resume, hoặc chọn output_dir mới.
subprocess.run(command, check=True)


## Lưu kết quả
Tải thư mục checkpoint, manifest, config, log CSV và TensorBoard từ Kaggle Output. Với train đầy đủ, chỉ đánh giá test sau khi chốt cấu hình trên val. Đổi `FULL_TRAINING=True`, `CLEAN_ROOT`, task và batch size theo VRAM.

In [ ]:
checkpoint = torch.load(Path(config["output_dir"])/"best.pt", map_location="cpu", weights_only=True)
print("epoch:", checkpoint["epoch"]+1, "val:", checkpoint["best_val"], "smoke_only:", checkpoint["smoke_only"])
